In [2]:
"""
=========================================================
FASE A — CREAR EL SUBCONJUNTO BALANCEADO DE 60 PACIENTES
=========================================================
Qué hace este script:
  1. Carga y fusiona las dos bases (ICBHI + Fraiwan).
  2. Extrae el identificador de paciente real.
  3. Selecciona 10 pacientes por clase (6 clases = 60 pacientes):
       - descarta pacientes con señal débil (RMS medio < 0.01)
       - de los válidos, coge los 10 con MÁS ventanas
  4. Asigna a cada paciente un fold (1..10), de forma que cada
     fold tenga 1 paciente de cada clase. SIN aleatoriedad.
  5. Aplica un TOPE de 30 ventanas por paciente, quedándose
     con las de mayor RMS (mejor calidad de señal).
  6. Guarda dos ficheros:
       - subconjunto_60.csv  : lista de los 60 pacientes y su fold
       - dataset_60_topado.csv : todas las ventanas finales a usar

CÓMO USARLO:
  Cambia las rutas path_icbhi y path_fraiwan a las de tu ordenador
  y ejecuta. Genera los dos CSV en la misma carpeta.
=========================================================
"""

import pandas as pd
import re

# =========================================================
# RUTAS — cámbialas a las de tu ordenador
# =========================================================
path_icbhi   = r"C:\Users\josem\Desktop\tfg\features_enriquecidas_corregido.csv"
path_fraiwan = r"C:\Users\josem\Desktop\tfg\features_fraiwan_limpio.csv"

# Parámetros de la selección
UMBRAL_RMS   = 0.01   # por debajo de esto, señal demasiado débil -> se descarta
N_POR_CLASE  = 10     # pacientes por clase
TOPE         = 30     # máximo de ventanas por paciente

# =========================================================
# 1. CARGAR Y FUSIONAR
# =========================================================
def cargar(path):
    return pd.read_csv(path, sep=';', decimal=',')

df_icbhi   = cargar(path_icbhi)
df_fraiwan = cargar(path_fraiwan)
df = pd.concat([df_icbhi, df_fraiwan], ignore_index=True)

# Limpieza básica
df = df.dropna(subset=['archivo'])
df = df[df['archivo'].astype(str).str.strip() != '']

# Normalizar nombres de clases
df['diagnostico'] = (df['diagnostico'].astype(str).str.strip().str.upper()
                     .replace({'HEALTHY': 'NORMAL', 'BRONCHIECTASIS': 'BRON'}))
clases = ['ASTHMA', 'BRON', 'COPD', 'HEART FAILURE', 'NORMAL', 'PNEUMONIA']
df = df[df['diagnostico'].isin(clases)]

# =========================================================
# 2. IDENTIFICADOR DE PACIENTE REAL
# =========================================================
def extraer_patient_id(archivo):
    archivo = str(archivo)
    m = re.match(r'^([A-Za-z]+)(\d+)_', archivo)   # Fraiwan: BP100_, DP100_, EP100_
    if m:
        return f'FRAIWAN_{m.group(2)}'
    m2 = re.match(r'^(\d+)_', archivo)             # ICBHI: 102_
    if m2:
        return f'ICBHI_{m2.group(1)}'
    return 'DESCONOCIDO'

df['pid'] = df['archivo'].apply(extraer_patient_id)

# =========================================================
# 3. SELECCIONAR 10 PACIENTES POR CLASE
# =========================================================
# Resumen por paciente: nº de ventanas y RMS medio
resumen = (df.groupby(['diagnostico', 'pid'])
             .agg(n_ventanas=('rms_mean', 'size'),
                  rms_medio=('rms_mean', 'mean'))
             .reset_index())

# Descartar pacientes con señal débil
resumen_validos = resumen[resumen['rms_medio'] >= UMBRAL_RMS]

# Para cada clase, los N con más ventanas, y asignar fold 1..N
seleccionados = []
for clase in clases:
    sub = (resumen_validos[resumen_validos['diagnostico'] == clase]
           .sort_values('n_ventanas', ascending=False))
    elegidos = sub.head(N_POR_CLASE).copy()
    elegidos['fold'] = range(1, len(elegidos) + 1)   # fold 1..10 (sin azar)
    seleccionados.append(elegidos)

sel = pd.concat(seleccionados).reset_index(drop=True)

# Guardar la lista de los 60 pacientes con su fold
sel[['pid', 'diagnostico', 'fold', 'n_ventanas', 'rms_medio']].to_csv(
    'subconjunto_60.csv', index=False, sep=';', decimal=',')

print("Pacientes seleccionados por clase:")
print(sel.groupby('diagnostico').size().to_string())
print("\nComprobación de folds (cada fold = 1 paciente por clase):")
print(sel.pivot_table(index='fold', columns='diagnostico',
                      values='pid', aggfunc='count', fill_value=0).to_string())

# =========================================================
# 4. APLICAR EL TOPE DE VENTANAS (las de mayor RMS)
# =========================================================
# Quedarnos solo con las ventanas de los 60 pacientes elegidos
df60 = df[df['pid'].isin(set(sel['pid']))].copy()

# Añadir la columna fold a cada ventana (según su paciente)
fold_de = dict(zip(sel['pid'], sel['fold']))
df60['fold'] = df60['pid'].map(fold_de)

# Aplicar el tope: por cada paciente, las TOPE ventanas de mayor RMS
trozos = []
for paciente, grupo in df60.groupby('pid'):
    grupo_ordenado = grupo.sort_values('rms_mean', ascending=False)
    trozos.append(grupo_ordenado.head(TOPE))
df60_topado = pd.concat(trozos, ignore_index=True)

# Guardar el dataset final
df60_topado.to_csv('dataset_60_topado.csv', index=False, sep=';', decimal=',')

print(f"\nVentanas antes del tope: {len(df60)}  ->  después: {len(df60_topado)}")
print("\nVentanas por clase tras el tope:")
print(df60_topado.groupby('diagnostico').size().sort_values(ascending=False).to_string())
print("\nFicheros generados:")
print("  - subconjunto_60.csv")
print("  - dataset_60_topado.csv")

Pacientes seleccionados por clase:
diagnostico
ASTHMA           10
BRON             10
COPD             10
HEART FAILURE    10
NORMAL           10
PNEUMONIA        10

Comprobación de folds (cada fold = 1 paciente por clase):
diagnostico  ASTHMA  BRON  COPD  HEART FAILURE  NORMAL  PNEUMONIA
fold                                                             
1                 1     1     1              1       1          1
2                 1     1     1              1       1          1
3                 1     1     1              1       1          1
4                 1     1     1              1       1          1
5                 1     1     1              1       1          1
6                 1     1     1              1       1          1
7                 1     1     1              1       1          1
8                 1     1     1              1       1          1
9                 1     1     1              1       1          1
10                1     1     1              1  